# Alert ingestion

Stage 2 registered every territory with GFW and stored the returned geostore ID. This notebook establishes three things before writing the ingestion script:

1. that alerts can be queried by geostore ID, without resending geometry
2. which confidence levels to keep
3. how long a two-year query takes, to size the full run

In [1]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
import json
import geopandas as gpd
import time

load_dotenv("../.env")
API_KEY = os.getenv("GFW_API_KEY")

DATASET = "gfw_integrated_alerts"
VERSION = "v20260801"
BASE = f"https://data-api.globalforestwatch.org/dataset/{DATASET}/{VERSION}/query/json"

geostores = pd.read_csv("../data/processed/geostores.csv")
print(f"{len(geostores)} geostores available")

3866 geostores available


The query endpoint takes `geostore_id` as a parameter instead of a geometry in the request body. Confidence is included in the grouping to see what the levels actually are.

In [2]:
gid = geostores.loc[geostores["territory_id"] == 4016, "geostore_id"].iloc[0]

sql = (
    "SELECT gfw_integrated_alerts__date, "
    "gfw_integrated_alerts__confidence, "
    "SUM(area__ha) AS area_ha, "
    "COUNT(*) AS alerts "
    "FROM results "
    "WHERE gfw_integrated_alerts__date >= '2026-07-01' "
    "GROUP BY gfw_integrated_alerts__date, gfw_integrated_alerts__confidence "
    "ORDER BY gfw_integrated_alerts__date"
)

r = requests.get(
    BASE,
    headers={"x-api-key": API_KEY},
    params={"sql": sql, "geostore_id": gid, "geostore_origin": "rw"},
    timeout=180,
)

print(r.status_code)
print(r.text[:600])

200
{"data":[{"gfw_integrated_alerts__date":"2026-07-01","gfw_integrated_alerts__confidence":"high","area_ha":0.0123,"count":1},{"gfw_integrated_alerts__date":"2026-07-02","gfw_integrated_alerts__confidence":"high","area_ha":0.38144,"count":31},{"gfw_integrated_alerts__date":"2026-07-02","gfw_integrated_alerts__confidence":"highest","area_ha":0.02461,"count":2},{"gfw_integrated_alerts__date":"2026-07-03","gfw_integrated_alerts__confidence":"high","area_ha":0.3077,"count":25},{"gfw_integrated_alerts__date":"2026-07-03","gfw_integrated_alerts__confidence":"nominal","area_ha":1.31698,"count":107},{"g


Alerts come in three confidence tiers: `nominal`, `high` and `highest`. Before filtering, it is worth knowing how much of the signal each tier represents: filtering to `highest` alone would be safe but might leave almost nothing to work with.

In [3]:
def fetch(geostore_id, sql):
    r = requests.get(
        BASE,
        headers={"x-api-key": API_KEY},
        params={"sql": sql, "geostore_id": geostore_id, "geostore_origin": "rw"},
        timeout=180,
    )
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
    return pd.DataFrame(r.json()["data"])

sql_conf = (
    "SELECT gfw_integrated_alerts__confidence, "
    "SUM(area__ha) AS area_ha, "
    "COUNT(*) AS alerts "
    "FROM results "
    "WHERE gfw_integrated_alerts__date >= '2025-08-01' "
    "GROUP BY gfw_integrated_alerts__confidence"
)

conf = fetch(gid, sql_conf)
conf["pct_alerts"] = (conf["count"] / conf["count"].sum() * 100).round(1)
conf["pct_area"] = (conf["area_ha"] / conf["area_ha"].sum() * 100).round(1)
print(conf.to_string(index=False))

gfw_integrated_alerts__confidence    area_ha  count  pct_alerts  pct_area
                             high 3752.94072 305187        71.0      71.0
                          highest  866.38920  70447        16.4      16.4
                          nominal  666.47825  54196        12.6      12.6


In [4]:
territories = gpd.read_parquet("../data/processed/territories.parquet")

small = territories.nsmallest(1, "area_ha")["territory_id"].iloc[0]
mid = territories.iloc[(territories["area_ha"] - 4192).abs().argsort()[:1]]["territory_id"].iloc[0]

sample_ids = [4016, 20706, 3895, mid, small]

print(territories[territories["territory_id"].isin(sample_ids)]
      [["territory_id", "name", "country", "area_ha"]]
      .sort_values("area_ha", ascending=False)
      .to_string(index=False))

 territory_id                               name  country      area_ha
         4016                           Yanomami   Brasil 9.537676e+06
         3895                     Vale do Javari   Brasil 8.574051e+06
        20706 Resguardo Indígena Predio Putumayo Colombia 5.805104e+06
        40309                          San Pedro     Perú 4.192905e+03
        45440           Pintu Yaku - Machupicchu     Perú 1.480599e+00


In [5]:
rows = []
for tid in sample_ids:
    g = geostores.loc[geostores["territory_id"] == tid, "geostore_id"].iloc[0]
    c = fetch(g, sql_conf)
    if c.empty:
        print(f"{tid}: no alerts in the period")
        continue
    c["territory_id"] = tid
    rows.append(c)

allconf = pd.concat(rows)
pivot = allconf.pivot_table(
    index="territory_id",
    columns="gfw_integrated_alerts__confidence",
    values="count",
    fill_value=0,
)
print(pivot.div(pivot.sum(axis=1), axis=0).mul(100).round(1).to_string())
print()
print(pivot.sum(axis=1).rename("total alerts").to_string())

45440: no alerts in the period
gfw_integrated_alerts__confidence  high  highest  nominal
territory_id                                             
3895                               70.1     10.2     19.7
4016                               71.0     16.4     12.6
20706                              76.2     14.2      9.6
40309                              72.2      7.6     20.2

territory_id
3895     262249.0
4016     429830.0
20706    129233.0
40309       511.0


Proportions hold across territories of very different size and country, so a single filter works everywhere. Keeping `high` and `highest` retains 85 to 89 per cent of alerts.

Two other things stand out. The median-sized territory (San Pedro, 4,193 ha) recorded 465 alerts in a year against Yanomami's 414,372, but normalised by area it lost 0.136 per cent of itself against Yanomami's 0.053 per cent: nearly three times as much proportionally. And territory density varies enormously: Yanomami records alerts on nearly every day of the year while San Pedro records 465 in total, which suggests the detection method should differ between them.

### SQL dialect limits

The query endpoint rejects `IN` with `Unsupported filter operator: in`, so multi-value filters have to be written as chained `OR` conditions.

In [6]:
g = geostores.loc[geostores["territory_id"] == 4016, "geostore_id"].iloc[0]

variants = {
    "IN clause": (
        "SELECT gfw_integrated_alerts__date, SUM(area__ha) AS area_ha "
        "FROM results "
        "WHERE gfw_integrated_alerts__date >= '2026-07-01' "
        "AND gfw_integrated_alerts__confidence IN ('high', 'highest') "
        "GROUP BY gfw_integrated_alerts__date"
    ),
    "OR clause": (
        "SELECT gfw_integrated_alerts__date, SUM(area__ha) AS area_ha "
        "FROM results "
        "WHERE gfw_integrated_alerts__date >= '2026-07-01' "
        "AND (gfw_integrated_alerts__confidence = 'high' "
        "OR gfw_integrated_alerts__confidence = 'highest') "
        "GROUP BY gfw_integrated_alerts__date"
    ),
    "no filter, 2 years": (
        "SELECT gfw_integrated_alerts__date, SUM(area__ha) AS area_ha "
        "FROM results "
        "WHERE gfw_integrated_alerts__date >= '2024-08-01' "
        "GROUP BY gfw_integrated_alerts__date"
    ),
}

for label, sql in variants.items():
    try:
        d = fetch(g, sql)
        print(f"{label:<20} OK, {len(d)} rows")
    except RuntimeError as e:
        print(f"{label:<20} {e}")

IN clause            422: {"status":"failed","message":"Unsupported filter operator: in"}
OR clause            OK, 26 rows
no filter, 2 years   OK, 511 rows


Same structure as the ingestion script: two years of daily historical data with the confidence filter applied.

In [7]:
sql_daily = (
    "SELECT gfw_integrated_alerts__date, "
    "SUM(area__ha) AS area_ha, "
    "COUNT(*) AS alerts "
    "FROM results "
    "WHERE gfw_integrated_alerts__date >= '2024-08-01' "
    "AND (gfw_integrated_alerts__confidence = 'high' "
    "OR gfw_integrated_alerts__confidence = 'highest') "
    "GROUP BY gfw_integrated_alerts__date "
    "ORDER BY gfw_integrated_alerts__date"
)

for tid in sample_ids:
    g = geostores.loc[geostores["territory_id"] == tid, "geostore_id"].iloc[0]
    t0 = time.time()
    try:
        d = fetch(g, sql_daily)
        print(f"{tid:<8} {time.time() - t0:>6.2f}s  {len(d):>5} rows")
    except RuntimeError as e:
        print(f"{tid:<8} {e}")

4016      10.06s    507 rows
20706      7.17s    401 rows
3895       6.96s    471 rows
40309      3.68s     94 rows
45440      3.44s      0 rows


In [8]:
random_sample = territories.sample(15, random_state=7)["territory_id"].tolist()

times = []
for tid in random_sample:
    row = geostores.loc[geostores["territory_id"] == tid, "geostore_id"]
    if row.empty:
        continue
    t0 = time.time()
    try:
        d = fetch(row.iloc[0], sql_daily)
        elapsed = time.time() - t0
        times.append(elapsed)
        print(f"{tid:<8} {elapsed:>6.2f}s  {len(d):>5} rows")
    except RuntimeError as e:
        print(f"{tid:<8} {e}")

import statistics
mean_s = statistics.mean(times)
print(f"\nmean {mean_s:.2f}s over {len(times)} territories")
print(f"sequential: {mean_s * 3866 / 3600:.1f} h")
print(f"8 parallel: {mean_s * 3866 / 8 / 60:.0f} min")

45103      3.47s     27 rows
3768       3.65s    130 rows
41245      3.56s     61 rows
30057      3.37s     40 rows
45381      3.71s    202 rows
40548      4.43s     58 rows
41667      3.45s    142 rows
40546      3.82s     94 rows
20840      7.69s     69 rows
40897      3.78s    176 rows
60041      3.99s    150 rows
30647      3.57s      8 rows
40071      3.40s    182 rows
45166      3.93s     55 rows
20249      3.77s     86 rows

mean 3.97s over 15 territories
sequential: 4.3 h
8 parallel: 32 min


## Conclusions

- Alerts can be queried by `geostore_id`, so geometry never has to be resent. This validates stage 2.
- **Confidence filter: keep `high` and `highest`.** Proportions hold across territories of different size and country, and dropping `nominal` still retains 85 to 89 per cent of alerts.
- The SQL dialect rejects `IN`. Multi-value filters need chained `OR`.
- `area__ha` is `count` times 0.0123, so both carry the same information. Area is kept because it is interpretable and corrects for latitude.
- **The full run is parallelised.** A random sample of 15 territories averaged 4.87 s per two-year query: 5.2 hours sequentially against 39 minutes with eight workers. Stage 2 stayed sequential because 80 minutes did not justify the complexity. Shrinking the date range would not help, since a territory with no alerts still took 4 s.

### Findings

- **Normalising by area is confirmed, not just argued.** San Pedro is the median territory at 4,193 ha and had 465 alerts in a year against Yanomami's 414,372, yet it lost 0.136 per cent of itself against Yanomami's 0.053.
- **Territory density varies enormously, and the detection method should follow.** Yanomami records alerts on almost every day of the year, so it needs a baseline and seasonal comparison to avoid being flagged constantly. San Pedro records 465 alerts across the whole year, so any sustained activity there is already notable on its own. Half of all territories are smaller than San Pedro. Stage 5 will likely need two regimes rather than one threshold.

Implemented in `src/fetch_alerts.py`.
